# eDRis Comprehensive Dataset Visualizations
This notebook generates all the high-end visuals you need for your SIH presentation across all your datasets (APTOS, IDRiD, DRIVE).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
print('Libraries loaded successfully! Ready to visualize.')

## 1. APTOS 2019: Class Imbalance
Proves the real-world medical data challenge.

In [ ]:
csv_path = '../datasets/classification/aptos2019/train.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    labels = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
    df['Diagnosis_Name'] = df['diagnosis'].map(labels)
    
    plt.figure(figsize=(10, 6))
    sns.countplot(x='Diagnosis_Name', data=df, order=labels.values(), hue='Diagnosis_Name', legend=False, palette='viridis')
    plt.title('APTOS 2019 Dataset: Diabetic Retinopathy Class Distribution', fontsize=16, fontweight='bold')
    plt.xlabel('Severity Level', fontsize=14)
    plt.ylabel('Number of Patients', fontsize=14)
    plt.show()
else:
    print(f'Cannot find CSV at {csv_path}')

## 2. APTOS 2019: Fundus Progression
Visualizing the 5 stages of Diabetic Retinopathy.

In [ ]:
img_dir = '../datasets/classification/aptos2019/train_images'
if os.path.exists(csv_path) and os.path.exists(img_dir):
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    
    for i, (level, name) in enumerate(labels.items()):
        sample_df = df[df['diagnosis'] == i]
        if not sample_df.empty:
            sample_img_id = sample_df.sample(1).iloc[0]['id_code']
            img_path = os.path.join(img_dir, f"{sample_img_id}.png")
            
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[i].imshow(img)
                axes[i].set_title(name, fontsize=16, fontweight='bold')
                axes[i].axis('off')
                
    plt.tight_layout()
    plt.show()


## 3. APTOS 2019: Lighting Variance (Why we use Preprocessing)
Judges will ask: *"Why did you write a preprocessing script?"* 
This RGB histogram proves that raw retinal images have terrible lighting, which is why your `preprocess_images.m` script (CLAHE) is so vital!

In [ ]:
if os.path.exists(img_dir):
    sample_img = os.listdir(img_dir)[0]
    img_path = os.path.join(img_dir, sample_img)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    color = ('r', 'g', 'b')
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.title('Raw Image (Dark/Low Contrast)', fontsize=16)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    for i, col in enumerate(color):
        histr = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
        plt.plot(histr, color=col)
        plt.xlim([0, 256])
    plt.title('RGB Color Distribution', fontsize=16)
    plt.xlabel('Pixel Intensity')
    plt.ylabel('Frequency')
    plt.show()


## 4. IDRiD: Ground Truth Segmentation Masks (For Layer 4 Explainability)
This shows that you have the medical ground-truth data (microaneurysms) to validate your White-Box Grad-CAM heatmaps later.

In [ ]:
idrid_img_dir = '../datasets/segmentation/idrid_segmentation/A. Segmentation/1. Original Images/a. Training Set'
idrid_mask_dir = '../datasets/segmentation/idrid_segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set'

if os.path.exists(idrid_img_dir) and os.path.exists(idrid_mask_dir):
    # Try to find a lesion mask folder (like Microaneurysms)
    mask_folders = glob.glob(os.path.join(idrid_mask_dir, '*'))
    if mask_folders:
        mask_folder = mask_folders[0] 
        masks = glob.glob(os.path.join(mask_folder, '*.tif'))
        
        if masks:
            mask_path = masks[0]
            # FIXED: Need to extract "IDRiD_01" not just "IDRiD" so the original image can be found!
            base = '_'.join(os.path.basename(mask_path).split('_')[:2])
            img_path = os.path.join(idrid_img_dir, base + '.jpg')
            
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                mask = cv2.imread(mask_path, 0)
                
                # Create a red overlay for the lesions
                overlay = img.copy()
                overlay[mask > 0] = [255, 0, 0] # Highlight in Red
                
                plt.figure(figsize=(18, 6))
                plt.subplot(1, 3, 1)
                plt.imshow(img)
                plt.title('IDRiD Original Image', fontsize=16)
                plt.axis('off')
                
                plt.subplot(1, 3, 2)
                plt.imshow(mask, cmap='gray')
                plt.title('Doctor\'s Lesion Mask', fontsize=16)
                plt.axis('off')
                
                plt.subplot(1, 3, 3)
                plt.imshow(overlay)
                plt.title('Lesions Highlighted', fontsize=16)
                plt.axis('off')
                plt.show()
            else:
                print("Could not find matching image:", img_path)
        else:
            print("No .tif masks found in:", mask_folder)
    else:
        print("No mask subfolders found.")
else:
    print('IDRiD path not found. Check your folder structure.')

## 5. DRIVE: Blood Vessel Segmentation
If you want to show the judges you can also map blood vessels to detect vascular decay, this is the graph for it!

In [ ]:
drive_img_dir = '../datasets/extra/drive_vessels/training/training/images'
drive_mask_dir = '../datasets/extra/drive_vessels/training/training/1st_manual'

if os.path.exists(drive_img_dir) and os.path.exists(drive_mask_dir):
    drive_imgs = glob.glob(os.path.join(drive_img_dir, '*.tif'))
    if drive_imgs:
        img_path = drive_imgs[0]
        base = os.path.basename(img_path).split('_')[0]
        mask_path = os.path.join(drive_mask_dir, base + '_manual1.gif')
        
        if os.path.exists(mask_path):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # PIL is better for reading .gif masks
            from PIL import Image
            mask = np.array(Image.open(mask_path))
            
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(img)
            plt.title('DRIVE Original Eye', fontsize=16)
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(mask, cmap='gray')
            plt.title('Vascular Tracing', fontsize=16)
            plt.axis('off')
            plt.show()
        else:
            print("Could not find matching mask:", mask_path)
else:
    print('DRIVE path not found.')